# **Actions units +random forrest **# 

In [ ]:

import os
import re
import joblib
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)
from sklearn.ensemble import RandomForestClassifier
import seaborn as sns
import matplotlib.pyplot as plt

# ========== CONFIGURABLE PATHS ==========
ROOT_DIR   = os.environ.get("AU_ROOT_DIR", "/kaggle/input/au-pemf-modified-pics/AU_pemf_modified_pics")
OUTPUT_DIR = Path(os.environ.get("WORK_DIR", "/kaggle/working"))
SPLIT_CSV  = os.environ.get("SPLIT_CSV", "/kaggle/input/mobilenetv3large93/mobile93/test_sequence_predictions.csv")
# ========== END CONFIG ==========

# ---------------- Parameters ----------------
CONFIDENCE_THRESHOLD = 0.8
RANDOM_STATE = 42
MODEL_OUT = OUTPUT_DIR / "rf_model.joblib"
SCALER_OUT = OUTPUT_DIR / "scaler.joblib"
TEST_RESULTS_CSV = OUTPUT_DIR / "rf_test_results.csv"

# ---------------- Utilities ----------------
def get_label_from_path(path):
    path_lower = str(path).lower()
    if "neutral" in path_lower:
        return 0
    elif any(pain_type.lower() in path_lower for pain_type in ["posed pain", "laser pain", "algometer pain"]):
        return 1
    else:
        return None

def extract_subject(path: str):
    m = re.search(r'(S\d{3})', str(path))
    return m.group(0) if m else None

def parse_sequence_path(path):
    """
    From a full path or relative path, return (subject_id, stimulus_type, sequence_name).
    This expects the last three folders encode those elements as in "S043/Posed Pain/BW frames".
    """
    p = os.path.normpath(str(path))
    parts = [pt for pt in p.split(os.sep) if pt != ""]
    if len(parts) >= 3:
        subject_id = parts[-3]
        stimulus_type = parts[-2]
        sequence_name = parts[-1]
    elif len(parts) == 2:
        subject_id = parts[-2]
        stimulus_type = parts[-1]
        sequence_name = ""
    elif len(parts) == 1:
        subject_id = parts[-1]
        stimulus_type = ""
        sequence_name = ""
    else:
        subject_id = ""
        stimulus_type = ""
        sequence_name = ""
    return subject_id, stimulus_type, sequence_name

def last3_dir_of_path(p):

    s = str(p).replace("\\", "/")
    parts = [pt for pt in s.split("/") if pt != ""]
    # If last part looks like a filename (contains a dot), drop it
    if parts and "." in parts[-1]:
        parts = parts[:-1]
    if len(parts) >= 3:
        return "/".join(parts[-3:])
    else:
        return "/".join(parts)

# ---------------- Feature extraction ----------------
records = []
au_set = set()

for subdir, _, files in os.walk(ROOT_DIR):
    for file in files:
        if not file.lower().endswith(".csv"):
            continue
        fp = os.path.join(subdir, file)
        label = get_label_from_path(fp)
        if label is None:
            continue
        try:
            df = pd.read_csv(fp)
            df.columns = df.columns.str.strip()
            if "success" in df.columns and "confidence" in df.columns:
                df = df[(df["success"] == 1) & (df["confidence"] > CONFIDENCE_THRESHOLD)]
            if df.empty:
                continue
            au_cols = [c for c in df.columns if "_r" in c]
            if not au_cols:
                continue
            au_set.update(au_cols)
            # store source filepath (full path to AU CSV)
            records.append({"source": fp, "df": df[au_cols].copy(), "label": label})
        except Exception as e:
            print("Failed reading:", fp, "error:", e)

if len(records) == 0:
    raise SystemExit("No valid samples found. Check ROOT_DIR and label extraction logic.")

au_list = sorted(au_set)
print(f"Collected {len(records)} samples; AU features count = {len(au_list)}")

features = []
labels = []
sources = []       # full file path to AU csv 
sources_dirs = []  # directory containing the AU csv 
subjects = []
for rec in records:
    df = rec["df"].reindex(columns=au_list)
    means = df.mean(axis=0).fillna(0).values
    stds  = df.std(axis=0).fillna(0).values
    vec = np.concatenate([means, stds])
    features.append(vec)
    labels.append(rec["label"])
    src_file = rec["source"]
    sources.append(src_file)
    src_dir = os.path.dirname(src_file)
    sources_dirs.append(src_dir)
    subj = extract_subject(src_file)
    subjects.append(subj if subj is not None else src_dir)

X = np.vstack(features)
y = np.array(labels)
sources = np.array(sources)
sources_dirs = np.array(sources_dirs)
subjects = np.array(subjects)

print("X shape:", X.shape, "class counts:", dict(zip(*np.unique(y, return_counts=True))))

# ---------------- Select test sequences using last-3-folder values from deep CSV ----------------
# Load split CSV
if not os.path.exists(SPLIT_CSV):
    raise SystemExit(f"SPLIT_CSV not found at: {SPLIT_CSV}")

split_df = pd.read_csv(SPLIT_CSV)
print(f"Loaded split CSV: {SPLIT_CSV} (rows: {len(split_df)})")

# Try to find a column that contains the relative sequence path (last-3 component)
candidate_cols = ['sequence_path', 'relative_path', 'video_path', 'path', 'sequence', 'seq_path', 'sequence_name', 'rel_last3']
found_col = None
for c in candidate_cols:
    if c in split_df.columns:
        found_col = c
        break

if found_col is None:
    # fallback: find any column with values that look like "S\d{3}/" in the sample rows
    for c in split_df.columns:
        sample_vals = split_df[c].dropna().astype(str).head(200).tolist()
        if any(re.search(r'S\d{3}[/\\]', v) for v in sample_vals):
            found_col = c
            break

if found_col is None:
    raise SystemExit(f"Could not infer sequence-path column in SPLIT_CSV. Columns: {list(split_df.columns)}. "
                     "Please provide SPLIT_CSV with a column like 'sequence_path' that contains values like 'S043/Posed Pain/BW frames'.")

print(f"Using column '{found_col}' from SPLIT_CSV to derive test sequence identifiers.")

# Normalize CSV entries to last-3 directory form
split_df['rel_last3'] = split_df[found_col].map(last3_dir_of_path)

# If the CSV includes a 'split' column, select only rows where split=='test' (case-insensitive)
if 'split' in split_df.columns:
    test_rows = split_df[split_df['split'].astype(str).str.lower() == 'test']
    test_set = set(test_rows['rel_last3'].dropna().unique())
else:
    test_set = set(split_df['rel_last3'].dropna().unique())

print(f"Test sequences referenced in CSV (unique last-3): {len(test_set)}")

# Build last-3 directory strings for the extracted sources 
sources_last3 = np.array([last3_dir_of_path(d) for d in sources_dirs])

# Mask samples whose last-3 dir appears in the test_set
is_test = np.isin(sources_last3, list(test_set))

print("Number of test samples found in inputs:", int(np.sum(is_test)))
if np.sum(is_test) == 0:
    #  show a few requested sequences vs available ones
    sample_requested = sorted(list(test_set))[:20]
    sample_available = sorted(list(set(sources_last3)))[:50]
    print("Requested (sample):", sample_requested)
    print("Available (sample):", sample_available)
    raise SystemExit("No test samples matched. Check that entries in the SPLIT_CSV and the extracted sources share the same last-3 directory format.")

# Show which requested sequences were not found among the available sources
missing_from_inputs = sorted(list(test_set - set(sources_last3)))
if missing_from_inputs:
    print(f"Warning: {len(missing_from_inputs)} sequences listed in SPLIT_CSV were not found among extracted sources. Example missing (up to 20):")
    for p in missing_from_inputs[:20]:
        print("  ", p)

# Build train/test sets
X_test, y_test, src_test, src_dirs_test, subj_test = X[is_test], y[is_test], sources[is_test], sources_dirs[is_test], subjects[is_test]
X_train, y_train, src_train, subj_train = X[~is_test], y[~is_test], sources[~is_test], subjects[~is_test]

print("Train size:", X_train.shape[0], "Test size:", X_test.shape[0])
print("Train class counts:", dict(zip(*np.unique(y_train, return_counts=True))))
print("Test class counts:", dict(zip(*np.unique(y_test, return_counts=True))))

# ---------------- Scale (fit on train only) ----------------
scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s  = scaler.transform(X_test)

# ---------------- Train Random Forest ----------------
model = RandomForestClassifier(
    class_weight='balanced',
    n_estimators=200,
    random_state=RANDOM_STATE
)

print("Fitting Random Forest...")
model.fit(X_train_s, y_train)

# ---------------- Evaluate & produce CSV with exact requested columns ----------------
if X_test.shape[0] > 0:
    y_pred = model.predict(X_test_s)

    # get probabilities for "pain" class 
    if hasattr(model, "predict_proba"):
        probs = model.predict_proba(X_test_s)  # shape (N, n_classes)
        if probs.ndim == 1:
            prob_pain = probs.ravel()
            prob_nopain = 1.0 - prob_pain
        else:
            # find index of class '1' in model.classes_
            try:
                idx_pain = int(np.where(model.classes_ == 1)[0])
            except Exception:
                idx_pain = 1 if probs.shape[1] > 1 else 0
            prob_pain = probs[:, idx_pain]
            if probs.shape[1] == 2:
                idx_nopain = 0 if idx_pain == 1 else 1
                prob_nopain = probs[:, idx_nopain]
            else:
                prob_nopain = 1.0 - prob_pain
    else:
        prob_pain = (y_pred == 1).astype(float)
        prob_nopain = 1.0 - prob_pain

    # print metrics
    acc = accuracy_score(y_test, y_pred)
    prec_macro = precision_score(y_test, y_pred, average='macro', zero_division=0)
    prec_weighted = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    rec_macro = recall_score(y_test, y_pred, average='macro', zero_division=0)
    rec_weighted = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1_macro = f1_score(y_test, y_pred, average='macro', zero_division=0)
    f1_weighted = f1_score(y_test, y_pred, average='weighted', zero_division=0)

    print(f"\nTest accuracy: {acc:.4f}")
    print(f"\nGeneral metrics:")
    print(f"  Precision (macro):   {prec_macro:.4f} / Precision (weighted):   {prec_weighted:.4f}")
    print(f"  Recall    (macro):   {rec_macro:.4f} / Recall    (weighted):   {rec_weighted:.4f}")
    print(f"  F1        (macro):   {f1_macro:.4f} / F1        (weighted):   {f1_weighted:.4f}")

    print("\nClassification report:")
    print(classification_report(y_test, y_pred, target_names=["Neutral", "Pain"]))

    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(6,5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Neutral","Pain"], yticklabels=["Neutral","Pain"])
    plt.title("Confusion Matrix - Random Forest")
    plt.xlabel("Predicted"); plt.ylabel("True")
    plt.show()

    # Build output rows matching screenshot columns
    rows = []
    for s_path, s_dir, subj, true_lbl, pred_lbl, p_pain, p_nop in zip(src_test, src_dirs_test, subj_test, y_test, y_pred, prob_pain, prob_nopain):
        # parse from directory (s_dir) not file path, so we get subject/stimulus/sequence correctly
        subject_id, stimulus_type, sequence_name = parse_sequence_path(s_dir)
        rows.append({
            "subject_id": subject_id,
            "stimulus_type": stimulus_type,
            "sequence_name": sequence_name,
            "sequence_path": last3_dir_of_path(s_dir),
            "true_label": int(true_lbl),
            "predicted_label": int(pred_lbl),
            "prob_pain": float(p_pain),
            "prob_nopain": float(p_nop)
        })

    test_results_df = pd.DataFrame(rows, columns=[
        "subject_id", "stimulus_type", "sequence_name", "sequence_path",
        "true_label", "predicted_label", "prob_pain", "prob_nopain"
    ])



    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    test_results_df.to_csv(TEST_RESULTS_CSV, index=False)
    print(f"Test predictions saved to {TEST_RESULTS_CSV}")

    # misclassified debug
    mis_idx = np.where(y_pred != y_test)[0]
    if mis_idx.size:
        print("\nMisclassified test samples:")
        for i in mis_idx:
            print(f"  {src_test[i]}  true={y_test[i]}  pred={y_pred[i]}")
    else:
        print("\nNo misclassifications on the test set.")
else:
    print("\nNo test samples found. Check SPLIT_CSV and that last-3 paths match your collected data.")

# ---------------- Persist scaler + model ----------------
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(scaler, SCALER_OUT)
joblib.dump(model, MODEL_OUT)
print(f"Saved scaler -> {SCALER_OUT}")
print(f"Saved model -> {MODEL_OUT}")

